# Asistente de derechos laborales: demo

Responde preguntas de empleados con la Ley Federal del Trabajo y el art. 123 constitucional,
y también con PDFs propios que subas (reglamento, contrato, lineamientos). Cada respuesta
cita de dónde viene cada dato y avisa cuando la respuesta no está en los documentos.

**Cómo ejecutarlo**

1. Desde la raíz del repo, con las llaves de OpenAI y Langfuse en `.env`:
   `poetry install` y luego `poetry run jupyter lab`.
2. Abre este notebook con el kernel **Python 3** del proyecto y corre las celdas en orden.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from src.mx_labor_law_assistant.assistant import answer
from src.mx_labor_law_assistant.user_docs import add_documents
from src.mx_labor_law_assistant.config import langfuse

print("Root y modulos cargados")

Root y modulos cargados


## 1. Pregunta sobre la ley

Una pregunta en lenguaje coloquial. `answer` busca los 5 fragmentos más parecidos, el modelo
responde citando cada dato y `validation` muestra qué fragmentos usó (`cited`), cuáles trajo la
búsqueda pero no usó (`not_cited`) y si todas las citas son de fragmentos entregados (`valid`).

In [2]:
resp = answer("Ya cumplí un año en mi trabajo, ¿cuántos días de vacaciones me tocan?")
display(Markdown(resp["answer"]))
resp["validation"]

Al cumplir tu primer año, te corresponden **al menos 12 días laborables de vacaciones pagadas**. En el régimen general, aumentan dos días por cada año subsecuente hasta llegar a 20; después, aumentan dos días por cada cinco años de servicios. (LFT, art. 76)

De esos días, al menos 12 deben disfrutarse de manera continua, aunque puedes distribuirlos de otra forma. Tu empleador debe concederte las vacaciones dentro de los seis meses siguientes a cumplir el año de servicios. (LFT, art. 78)(LFT, art. 81)

Si trabajas de forma discontinua o por temporada, el periodo se calcula en proporción a los días trabajados durante el año. (LFT, art. 77)

{'cited': {'Fragmento 1': 'LFT, art. 76',
  'Fragmento 3': 'LFT, art. 78',
  'Fragmento 4': 'LFT, art. 81',
  'Fragmento 5': 'LFT, art. 77'},
 'not_cited': {'Fragmento 2': 'LFT, art. 199'},
 'unknown': [],
 'valid': True}

## 2. Cargar documentos propios

`add_documents` recibe uno o varios PDFs, los parte por página y los guarda en
`data/user.sqlite`, aparte de las leyes. Muestra cuáles se procesaron (`done`) y cuáles
fallaron (`failed`); si uno falla, los demás siguen. Aquí se carga un documento de ejemplo:
los lineamientos de conciliación laboral de la CDMX.

In [3]:
add_documents([ROOT / "data" / "examples" / "lineamientos_ccl_cdmx.pdf"])

{'done': ['lineamientos_ccl_cdmx.pdf'], 'failed': []}

## 3. Pregunta sobre la ley y el documento

Ahora la búsqueda incluye las leyes y el documento cargado. Si los dos hablan del mismo tema,
la respuesta dice qué dice cada uno y de cuál viene cada dato: el documento se cita con su
nombre de archivo y página, la ley con su artículo.

In [4]:
resp = answer("¿Cuántos días puede durar como máximo la conciliación antes de demandar?")
display(Markdown(resp["answer"]))
resp["validation"]

Según los lineamientos del Centro de Conciliación Laboral de la Ciudad de México, el procedimiento prejudicial no debe exceder **45 días naturales**, contados desde la presentación de la solicitud. (lineamientos_ccl_cdmx.pdf, p. 4)

La LFT citada establece plazos para audiencias —por ejemplo, que la audiencia se efectúe dentro de los 15 días siguientes y que una nueva audiencia, cuando proceda, se celebre dentro de los cinco días siguientes—, pero los fragmentos de la LFT no indican un plazo máximo total de duración. (LFT, art. 684-E)(LFT, art. 684-E)

{'cited': {'Fragmento 1': 'LFT, art. 684-E',
  'Fragmento 2': 'LFT, art. 684-E',
  'Fragmento 3': 'lineamientos_ccl_cdmx.pdf, p. 4'},
 'not_cited': {'Fragmento 4': 'LFT, art. 684-E',
  'Fragmento 5': 'LFT, art. 48'},
 'unknown': [],
 'valid': True}

## 4. Pregunta sin respuesta

El monto del salario mínimo no está en la ley ni en el documento, porque lo fija cada año
la Comisión Nacional de los Salarios Mínimos. El asistente debe decir que no está en los
documentos en lugar de inventarlo. `langfuse.flush()` envía las trazas antes de cerrar.

In [5]:
resp = answer("¿Cuánto es el salario mínimo diario vigente este año en mi zona?")
display(Markdown(resp["answer"]))
langfuse.flush()

Los fragmentos no indican el monto diario vigente para este año ni el área geográfica de aplicación, así que no permiten determinar la cifra. La ley establece que los salarios mínimos generales rigen según el área geográfica correspondiente; para ciertos trabajos también se fijan salarios mínimos profesionales. (LFT, art. 92)(LFT, art. 335)(LFT, art. 280 Bis)